In [1]:
from opfython.models import SupervisedOPF

import logging
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr


In [2]:
from testflows.combinatorics import Covering
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn import svm
import random
from sklearn.neighbors import KNeighborsClassifier
from numpy import inf
import time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import umap
from joblib import Parallel, delayed
import logging
import sys
import warnings
import os
from contextlib import redirect_stdout, redirect_stderr

# Suppress all logging output
os.environ['OPF_LOG_LEVEL'] = 'ERROR'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'

warnings.filterwarnings('ignore')

# Suppress all loggers - disable INFO and DEBUG
logging.getLogger().setLevel(logging.WARNING)
logging.disable(logging.INFO)

# Disable specific loggers
for logger_name in ['opfython', 'opfython.core', 'opfython.models', 'opfython.ml']:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.WARNING)
    logger.disabled = True
    logger.propagate = False
    logger.handlers.clear()
    logger.addHandler(logging.NullHandler())

/opt/miniconda3/envs/your_env_name/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')
df_algarrobo = pd.read_csv(r'../data/data_temp/algarrobo.csv')
df_fruits_pures = pd.read_csv(r'../data/data_temp/MIR_Fruit_purees.csv')
df_fresh_meat = pd.read_csv(r'../data/data_temp/Fresh_meats.csv')
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')

df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]
X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())

unique_names_berry = df_fruits_pures["label"].unique()
fruits_pures_x = df_fruits_pures.iloc[:,1:]
y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=unique_names_berry, value =range(0,len(unique_names_berry)))
X_fruit_puree = (fruits_pures_x-fruits_pures_x.min())/(fruits_pures_x.max()-fruits_pures_x.min())


unique_names_meat = df_fresh_meat["meat"].unique()
meat_x = df_fresh_meat.iloc[:,4:]
y_meat = df_fresh_meat.iloc[:,0:1].replace(to_replace=unique_names_meat,value=range(0,len(unique_names_meat)))
X_meat =  (meat_x-meat_x.min())/(meat_x.max()-meat_x.min())

unique_names_olive = df_olive["Provenance"].unique()
olive_x = df_olive.iloc[:,3:]
y_olive = df_olive.iloc[:,2:3].replace(to_replace=unique_names_olive,value=range(0,len(unique_names_olive)))
X_olive =  (olive_x-olive_x.min())/(olive_x.max()-olive_x.min())


In [4]:
from sklearn.model_selection import KFold

def ICAFS(dataset_X, dataset_Y, strenght,max_iteartion,clasifier, print_logs=False):

  max_it = 1 
  std_list = []
  iter_list = []
  score_list = []
  featur_list = []
  
  v_variable = [0, 1]
  best_f1_score = float('-inf')
  max_iteartion_aux =  max_iteartion
  best_data_set = dataset_X.columns.values.copy()

  #initial_test_training(score_list,iter_list,featur_list,dataset_X,dataset_Y,clasifier)
  while max_iteartion_aux > 0:

      
      partial_score = 0
      dict_parameters = {}
      partial_best_list = []
      subsets_to_consider = []

      for colum_key in best_data_set:
          dict_parameters[colum_key] = v_variable
      random.shuffle(best_data_set)
      generate_covering_array = Covering(dict_parameters, strength=strenght)
      for i, test in enumerate(generate_covering_array.array):
          list_attributes_to_consider = []

          check_for_all_cero = True
          for (test_key, test_value) in test.items():
              if test_value == 1:
                  check_for_all_cero = False
                  list_attributes_to_consider.append(test_key)

          if check_for_all_cero:
              continue  
          subsets_to_consider.append((list_attributes_to_consider, i))
      
      results = Parallel(n_jobs=-1,backend='loky')(delayed(run_cv)(clasifier, dataset_X, subset_features, dataset_Y.values.ravel(), i) for subset_features,i in subsets_to_consider)   
      sorted_results = sorted(results, key=lambda x: x[2])
      
      for score, std, i, subset_features in sorted_results:
        if score >= partial_score:
              partial_score = score
              best_std = std
              partial_best_list = subset_features.copy()

      best_data_set = partial_best_list.copy()
      best_f1_score = partial_score
      
      if print_logs:
        print(f"best f1 score= {best_f1_score}, iteration:{max_it}, numbers features selected ={ len(best_data_set)},best features selected={', '.join(best_data_set)}" )

      iter_list.append(max_it)
      aux_data_score = best_f1_score
      score_list.append(aux_data_score)

      max_iteartion_aux = max_iteartion_aux-1
      max_it = max_it +1
      featur_list.append(len(best_data_set))
      std_list.append(best_std)

  return iter_list,featur_list,score_list,std_list

def initial_test_training(score_list, iter_list, feature_list ,X,y,clasifier):
     
     X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(X.values, y.values.ravel(), test_size=0.20, random_state=42)
     clasifier.fit(X_train_temp, y_train_temp)
     y_pred = clasifier.predict(X_test_temp)
     score_list.append(f1_score(y_test_temp, y_pred,average='macro'))
     feature_list.append(X.shape[1])
     iter_list.append(0)

def kfold_with_opf_in_icafs(clasifier,x_data, y_data):
    
    scores = []
    kf = KFold(n_splits=5,shuffle=True, random_state =42)
    for i, (train_index, test_index) in enumerate(kf.split(x_data)):
   
        x_fold_train = x_data[train_index,:]
        y_fold_train = y_data[train_index]

        x_fold_test =  x_data[test_index,:]
        y_fold_test = y_data[test_index]
        
        clasifier.fit(x_fold_train, y_fold_train)
        y_pred_fold = clasifier.predict(x_fold_test)
        score = f1_score(y_fold_test, y_pred_fold,average='macro')
        scores.append(score)

    return np.asarray(scores, dtype=np.float32)

def run_cv(model, X, subset_features, y, index):
    
    # Create null file to suppress all output
    null_file = StringIO()
    # Disable INFO and DEBUG logging temporarily
    logging.disable(logging.INFO)
    
    try:
        # Redirect both stdout and stderr to null
        with redirect_stdout(null_file), redirect_stderr(null_file):
            scores = kfold_with_opf_in_icafs(model, X[subset_features].values, y)
        return (scores.mean(), scores.std(), index, subset_features)
    finally:
        # Restore logging state
        logging.disable(logging.NOTSET)

In [5]:
def  plot_results_for_covering_array(scores,feature,num_of_iterarion, path_to_save_image):
        color = 'tab:blue'
        res_scores = np.array(scores)
        res_features = np.array(feature)
        res_iter = np.array(num_of_iterarion)

        plt.figure(figsize=(11, 10))
        
        fig, ax1 = plt.subplots()
        barwidth = 0.4
        color = 'tab:red'
        ax1.set_xlabel('Iterations')
        ax1.set_ylabel('Number of features', color=color)
        #ax1.set_title("ICAFS Feature selection on the Cacao dataset")
        ax1.spines['top'].set_visible(False)
        ax1.bar(res_iter-0.2, res_features, color=color, width=barwidth)
        ax1.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax1.set_ylim(1,max(res_features)+3)
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
        
        for bar in ax1.patches:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height}', fontsize=10,
                    ha='center', va='bottom', rotation=90)
            
        ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
        color = 'tab:blue'
        ax2.set_ylabel('F1_score', color=color)
        ax2.bar(res_iter+0.2, res_scores, color=color, width=barwidth)
        ax2.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax2.set_ylim(min(res_scores)-0.001, max(res_scores)+0.001)

        fig.tight_layout()  # otherwise the right y-label is slightly clipped
       
        for bar in ax2.patches:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height:.2f}', fontsize=10,
                    ha='center', va='bottom', rotation=90)

        #plt.title('ICAFS Feature selection for Cacao Dataset with OPF', y=-0.20)
        plt.gca().set_frame_on(False)
        plt.savefig(path_to_save_image)

In [20]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score ,std_list = ICAFS(X_cacao,y_cacao,2,10,algorithm,True )
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cacao_opf_icafs.png')

best f1 score= 0.9387187957763672, iteration:1, numbers features selected =967,best features selected=1107, 1532, 1533, 1534, 1535, 1536, 1537, 1538, 1539, 1540, 1541, 1542, 1543, 1544, 1545, 1546, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1555, 1556, 1557, 1558, 1559, 1560, 1561, 1562, 1563, 1564, 1565, 1566, 1567, 1568, 1569, 1570, 1571, 1572, 1573, 1574, 1575, 1576, 1577, 1578, 1579, 1580, 1581, 1582, 1583, 1584, 1585, 1586, 1587, 1588, 1589, 1590, 1591, 1592, 1593, 1594, 1596, 1597, 1598, 1599, 1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609, 1610, 1612, 1613, 1614, 1615, 1616, 1617, 1618, 1619, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1627, 1628, 1629, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1649, 1650, 1651, 1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661, 1662, 1663, 1664, 1665, 1666, 1667, 1668, 1669, 1670, 1671, 1672, 1673, 1674, 1675, 1676, 1677, 1678, 1679, 1680, 1681, 1682, 1683

In [21]:
print("ICAFS OPF - Cacao Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Cacao, Std: {std:.6f}")

ICAFS OPF - Cacao Dataset
Index 0: Dataset Cacao, Std: 0.021387
Index 1: Dataset Cacao, Std: 0.019718
Index 2: Dataset Cacao, Std: 0.018710
Index 3: Dataset Cacao, Std: 0.018710
Index 4: Dataset Cacao, Std: 0.017953
Index 5: Dataset Cacao, Std: 0.016966
Index 6: Dataset Cacao, Std: 0.011451
Index 7: Dataset Cacao, Std: 0.015307
Index 8: Dataset Cacao, Std: 0.015341
Index 9: Dataset Cacao, Std: 0.018174


In [22]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = ICAFS(X_algarrobo,y_algarrobo,2,10,algorithm,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\algarrobo_opf_icafs.png')

best f1 score= 0.7745554447174072, iteration:1, numbers features selected =9,best features selected=NGRDI, NDVI, RVI, DVI, EVI, REVI, NDRE, RERVI, REDVI
best f1 score= 0.72181236743927, iteration:2, numbers features selected =6,best features selected=NGRDI, NDVI, RVI, EVI, NDRE, REDVI
best f1 score= 0.7224792242050171, iteration:3, numbers features selected =4,best features selected=NGRDI, NDVI, RVI, NDRE
best f1 score= 0.7280455231666565, iteration:4, numbers features selected =3,best features selected=NGRDI, NDVI, RVI
best f1 score= 0.7280455231666565, iteration:5, numbers features selected =3,best features selected=NGRDI, NDVI, RVI
best f1 score= 0.7280455231666565, iteration:6, numbers features selected =3,best features selected=NGRDI, NDVI, RVI
best f1 score= 0.7280455231666565, iteration:7, numbers features selected =3,best features selected=NGRDI, NDVI, RVI
best f1 score= 0.7280455231666565, iteration:8, numbers features selected =3,best features selected=NGRDI, NDVI, RVI
best f

In [23]:
print("ICAFS OPF - Algarrobo Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Algarrobo, Std: {std:.6f}")

ICAFS OPF - Algarrobo Dataset
Index 0: Dataset Algarrobo, Std: 0.031493
Index 1: Dataset Algarrobo, Std: 0.018907
Index 2: Dataset Algarrobo, Std: 0.010015
Index 3: Dataset Algarrobo, Std: 0.036395
Index 4: Dataset Algarrobo, Std: 0.036395
Index 5: Dataset Algarrobo, Std: 0.036395
Index 6: Dataset Algarrobo, Std: 0.036395
Index 7: Dataset Algarrobo, Std: 0.036395
Index 8: Dataset Algarrobo, Std: 0.036395
Index 9: Dataset Algarrobo, Std: 0.036395


In [6]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = ICAFS(X_cis,y_cis,3,10,algorithm,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'./output_images/cis_opf_icafs.png')

best f1 score= 0.9809911847114563, iteration:1, numbers features selected =4,best features selected=X, Y, X40, Y40
best f1 score= 0.9924972653388977, iteration:2, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:3, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:4, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:5, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:6, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:7, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:8, numbers features selected =3,best features selected=X, Y, Y40
best f1 score= 0.9924972653388977, iteration:9, numbers features selected =3,best features selected=X, Y, Y40
best 

In [7]:
print("CAFS OPF - CIS Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset CIS, Std: {std:.6f}")

CAFS OPF - CIS Dataset
Index 0: Dataset CIS, Std: 0.002472
Index 1: Dataset CIS, Std: 0.002099
Index 2: Dataset CIS, Std: 0.002099
Index 3: Dataset CIS, Std: 0.002099
Index 4: Dataset CIS, Std: 0.002099
Index 5: Dataset CIS, Std: 0.002099
Index 6: Dataset CIS, Std: 0.002099
Index 7: Dataset CIS, Std: 0.002099
Index 8: Dataset CIS, Std: 0.002099
Index 9: Dataset CIS, Std: 0.002099


In [24]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = ICAFS(X_fruit_puree,y_fruit_puree,2,10,algorithm,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'./output_images/fruit_puree_opf_icafs.png')

best f1 score= 0.9603063464164734, iteration:1, numbers features selected =188,best features selected=907.047, 1080.747, 1088.467, 1092.327, 1096.187, 1100.047, 1103.907, 1107.767, 1111.627, 1115.487, 1119.347, 1123.207, 1127.067, 1130.927, 1134.786, 1138.646, 1142.506, 1146.366, 1150.226, 1154.086, 1157.946, 1161.806, 1165.666, 1169.526, 1173.386, 1177.246, 1181.106, 1184.966, 1188.826, 1192.686, 1196.546, 1200.406, 1204.266, 1208.126, 1211.986, 1215.846, 1219.706, 1223.566, 1227.426, 1231.286, 1235.146, 1239.006, 1242.866, 1246.726, 1250.586, 1254.446, 1258.306, 1262.166, 1266.026, 1269.886, 1273.746, 1277.606, 1281.466, 1285.326, 1289.186, 1293.046, 1296.906, 1300.766, 1304.626, 1308.486, 1312.346, 1316.206, 1320.066, 1323.926, 1327.786, 1331.646, 1335.506, 1339.366, 1343.226, 1347.086, 1350.946, 1354.806, 1358.666, 1362.526, 1366.386, 1370.246, 1374.106, 1377.966, 1381.826, 1385.686, 1389.545, 1393.405, 1397.265, 1401.125, 1404.985, 1408.845, 1412.705, 1416.565, 1420.425, 1424.285,

In [25]:
print("ICAFS OPF - Fruit Puree Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Fruit Puree, Std: {std:.6f}")

ICAFS OPF - Fruit Puree Dataset
Index 0: Dataset Fruit Puree, Std: 0.009989
Index 1: Dataset Fruit Puree, Std: 0.010333
Index 2: Dataset Fruit Puree, Std: 0.011920
Index 3: Dataset Fruit Puree, Std: 0.012863
Index 4: Dataset Fruit Puree, Std: 0.015032
Index 5: Dataset Fruit Puree, Std: 0.009709
Index 6: Dataset Fruit Puree, Std: 0.011622
Index 7: Dataset Fruit Puree, Std: 0.009285
Index 8: Dataset Fruit Puree, Std: 0.011530
Index 9: Dataset Fruit Puree, Std: 0.013584


In [26]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = ICAFS(X_meat,y_meat,2,10,algorithm,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'./output_images/meat_opf_icafs.png')

best f1 score= 0.9408732652664185, iteration:1, numbers features selected =336,best features selected=1018.856, 1221.4595, 1223.389, 1225.3195, 1227.249, 1229.1785, 1231.108, 1233.0375, 1234.967, 1236.8965, 1238.826, 1240.7555, 1242.685, 1244.6145, 1246.544, 1248.4745, 1252.3335, 1254.263, 1256.1925, 1258.122, 1260.0515, 1261.981, 1263.9105, 1265.84, 1267.7695, 1269.699, 1271.6285, 1273.558, 1275.4875, 1277.417, 1279.3465, 1281.276, 1283.2065, 1285.136, 1287.0655, 1288.995, 1290.9245, 1292.854, 1294.7835, 1296.713, 1298.6425, 1300.572, 1302.5015, 1304.431, 1306.3615, 1308.291, 1310.2205, 1312.15, 1314.0795, 1316.009, 1317.9385, 1319.868, 1321.7975, 1323.727, 1325.6565, 1327.586, 1329.5155, 1331.445, 1333.3745, 1335.304, 1337.2335, 1339.163, 1341.0935, 1343.023, 1344.9525, 1346.882, 1348.8115, 1350.741, 1352.6705, 1354.6, 1356.5295, 1358.459, 1360.3885, 1362.318, 1364.2485, 1366.178, 1368.1075, 1370.037, 1371.9665, 1373.896, 1375.8255, 1377.755, 1379.6845, 1381.614, 1383.5435, 1385.473,

In [27]:
print("ICAFS OPF - Meat Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Meat, Std: {std:.6f}")

ICAFS OPF - Meat Dataset
Index 0: Dataset Meat, Std: 0.024809
Index 1: Dataset Meat, Std: 0.034485
Index 2: Dataset Meat, Std: 0.018668
Index 3: Dataset Meat, Std: 0.018668
Index 4: Dataset Meat, Std: 0.027452
Index 5: Dataset Meat, Std: 0.027402
Index 6: Dataset Meat, Std: 0.034101
Index 7: Dataset Meat, Std: 0.022566
Index 8: Dataset Meat, Std: 0.022566
Index 9: Dataset Meat, Std: 0.073455


In [28]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = ICAFS(X_olive,y_olive,2,10,algorithm,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'./output_images/olive_opf_icafs.png')

best f1 score= 0.9040400385856628, iteration:1, numbers features selected =367,best features selected=812.3985, 860.636, 862.5655, 864.495, 866.4245, 868.354, 870.2835, 872.213, 874.1425, 876.072, 878.0015, 879.931, 881.8605, 883.79, 885.7195, 887.649, 889.5785, 891.508, 893.4375, 895.367, 897.2965, 899.226, 901.1555, 903.085, 905.0145, 906.944, 908.8735, 910.803, 912.7325, 914.662, 916.5915, 918.521, 920.4505, 922.38, 924.3095, 926.239, 928.1685, 930.098, 932.0275, 933.957, 935.8865, 937.816, 939.7455, 941.675, 943.6045, 945.534, 947.4645, 949.394, 951.3235, 953.253, 955.1825, 957.112, 959.0415, 960.971, 962.9005, 964.83, 966.7595, 968.689, 970.6185, 972.548, 974.4775, 976.407, 978.3365, 980.266, 982.1955, 984.125, 986.0545, 987.984, 989.9135, 991.843, 993.7725, 995.702, 997.6315, 999.561, 1001.4905, 1003.42, 1005.3495, 1007.279, 1009.2085, 1011.138, 1013.0675, 1014.997, 1016.9265, 1018.856, 1020.7855, 1022.715, 1024.6445, 1026.574, 1028.5035, 1030.433, 1032.3625, 1034.292, 1036.2215,

In [30]:
print("ICAFS OPF - Olive Tree Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Olive Tree, Std: {std:.6f}")

ICAFS OPF - Olive Tree Dataset
Index 0: Dataset Olive Tree, Std: 0.031905
Index 1: Dataset Olive Tree, Std: 0.041484
Index 2: Dataset Olive Tree, Std: 0.041880
Index 3: Dataset Olive Tree, Std: 0.041880
Index 4: Dataset Olive Tree, Std: 0.041880
Index 5: Dataset Olive Tree, Std: 0.044473
Index 6: Dataset Olive Tree, Std: 0.047776
Index 7: Dataset Olive Tree, Std: 0.065772
Index 8: Dataset Olive Tree, Std: 0.042189
Index 9: Dataset Olive Tree, Std: 0.037232
